# Chapter 1.5: Latency vs Throughput -- Serving Metrics

## Learning Objectives

By the end of this notebook, you will:

1. **Define** key LLM serving metrics: TTFT, TPOT, ITL, E2E latency, throughput
2. **Explain** what affects each metric and the trade-offs between them
3. **Understand** how batch size impacts latency and throughput
4. **Simulate** and plot latency distributions
5. **Set** appropriate SLOs (Service Level Objectives) for LLM serving
6. **Measure** metrics correctly using proper benchmarking methodology

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part1/chapter_1.5_metrics.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part1/chapter_1.5_metrics.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import time
import heapq
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field

np.random.seed(42)
print("Ready to explore serving metrics!")

---

## 1. Key Metrics for LLM Serving

### 1.1 Time To First Token (TTFT)

$$\text{TTFT} = t_{\text{first token}} - t_{\text{request arrival}}$$

**What it measures**: How long the user waits before seeing the first token of the response.

**What affects it**:
- Prompt length (prefill computation time)
- Queue wait time (if the system is busy)
- Model size and GPU speed
- Batch scheduling decisions

**Typical values**: 50ms - 2s for interactive applications

### 1.2 Time Per Output Token (TPOT) / Inter-Token Latency (ITL)

$$\text{TPOT} = \frac{t_{\text{last token}} - t_{\text{first token}}}{n_{\text{output tokens}} - 1}$$

**What it measures**: Average time between consecutive output tokens.

**What affects it**:
- Batch size (more concurrent requests = higher TPOT per request)
- KV-cache length (longer sequences = more data to read)
- GPU memory bandwidth
- Quantization level

**Typical values**: 10ms - 100ms (human reading speed is ~250ms/word ~ 50ms/token)

### 1.3 End-to-End Latency

$$\text{E2E Latency} = \text{TTFT} + \text{TPOT} \times (n_{\text{output tokens}} - 1)$$

### 1.4 Throughput

$$\text{Throughput} = \frac{\text{Total output tokens generated}}{\text{Total time}} \quad [\text{tokens/s}]$$

In [ ]:
## Figure: Single Request Timeline with Labeled Intervals
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(18, 5))

BLUE, GREEN, ORANGE, RED, PURPLE = '#4A90D9', '#27AE60', '#F39C12', '#E74C3C', '#8E44AD'

# Timeline parameters
queue_t = 0.15
prefill_t = 0.35
ttft = queue_t + prefill_t
tpot = 0.025
n_tokens = 15
y = 1.0
h = 0.5

# Queue
ax.barh(y, queue_t, left=0, height=h, color='#95A5A6', alpha=0.7, edgecolor='black')
ax.text(queue_t/2, y, 'Queue\nWait', ha='center', va='center', fontsize=8, fontweight='bold')

# Prefill
ax.barh(y, prefill_t, left=queue_t, height=h, color=BLUE, alpha=0.8, edgecolor='black')
ax.text(queue_t + prefill_t/2, y, 'Prefill\n(Process prompt)', ha='center', va='center', fontsize=8, fontweight='bold', color='white')

# Decode tokens
for i in range(n_tokens):
    start = ttft + i * tpot
    c = GREEN if i % 2 == 0 else '#2ECC71'
    ax.barh(y, tpot*0.9, left=start, height=h, color=c, alpha=0.8, edgecolor='black', linewidth=0.5)
    if i < 5 or i == n_tokens-1:
        ax.text(start + tpot/2, y, f't{i}', ha='center', va='center', fontsize=6)

last_token_end = ttft + n_tokens * tpot

# TTFT bracket
ax.annotate('', xy=(0, y - 0.45), xytext=(ttft, y - 0.45),
            arrowprops=dict(arrowstyle='<->', color=RED, lw=2.5))
ax.text(ttft/2, y - 0.65, f'TTFT = {ttft*1000:.0f} ms', ha='center', fontsize=11,
        fontweight='bold', color=RED)

# ITL bracket  
itl_start = ttft + tpot
itl_end = ttft + 2*tpot
ax.annotate('', xy=(ttft, y + 0.45), xytext=(ttft + tpot, y + 0.45),
            arrowprops=dict(arrowstyle='<->', color=ORANGE, lw=2))
ax.text(ttft + tpot/2, y + 0.6, f'ITL = {tpot*1000:.0f} ms', ha='center', fontsize=9,
        fontweight='bold', color=ORANGE)

# E2E bracket
ax.annotate('', xy=(0, y + 0.8), xytext=(last_token_end, y + 0.8),
            arrowprops=dict(arrowstyle='<->', color=PURPLE, lw=2.5))
ax.text(last_token_end/2, y + 1.0, f'End-to-End Latency = {last_token_end*1000:.0f} ms', 
        ha='center', fontsize=12, fontweight='bold', color=PURPLE)

# TPOT bracket
decode_start = ttft
ax.annotate('', xy=(decode_start, y - 0.85), xytext=(last_token_end, y - 0.85),
            arrowprops=dict(arrowstyle='<->', color=GREEN, lw=2))
ax.text((decode_start + last_token_end)/2, y - 1.05,
        f'Decode Phase: {n_tokens} tokens, TPOT = {tpot*1000:.0f} ms/token',
        ha='center', fontsize=10, fontweight='bold', color=GREEN)

# Markers
ax.axvline(x=0, color='black', linestyle='-', lw=1, alpha=0.5)
ax.text(0, y + 1.4, 'Request\nArrives', ha='center', fontsize=8, fontweight='bold')
ax.axvline(x=ttft, color=RED, linestyle='--', lw=1.5, alpha=0.5)
ax.text(ttft, y + 1.4, 'First\nToken', ha='center', fontsize=8, fontweight='bold', color=RED)
ax.axvline(x=last_token_end, color=PURPLE, linestyle='--', lw=1.5, alpha=0.5)
ax.text(last_token_end, y + 1.4, 'Last\nToken', ha='center', fontsize=8, fontweight='bold', color=PURPLE)

ax.set_xlabel('Time (seconds)', fontsize=12)
ax.set_title('Anatomy of an LLM Request: Key Latency Metrics', fontsize=15, fontweight='bold')
ax.set_yticks([])
ax.set_xlim(-0.05, last_token_end + 0.05)
ax.set_ylim(-1.3, 1.8)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()
print("TTFT = Time to First Token (queue + prefill). Critical for perceived responsiveness.")
print("ITL/TPOT = Inter-Token Latency / Time Per Output Token. Determines streaming speed.")
print("E2E = End-to-End latency = TTFT + (n_output_tokens - 1) * TPOT")

In [ ]:
# Visualize the timeline of a single LLM request

fig, ax = plt.subplots(figsize=(16, 4))

# Timeline
queue_time = 0.2  # seconds
ttft = 0.5        # prefill time
tpot = 0.03       # 30ms per token
n_tokens = 20

# Draw timeline
y = 0.5
height = 0.3

# Queue time
ax.barh(y, queue_time, left=0, height=height, color='#95a5a6', alpha=0.7, edgecolor='black')
ax.text(queue_time/2, y, 'Queue', ha='center', va='center', fontsize=10, fontweight='bold')

# Prefill
ax.barh(y, ttft, left=queue_time, height=height, color='#3498db', alpha=0.7, edgecolor='black')
ax.text(queue_time + ttft/2, y, 'Prefill\n(TTFT)', ha='center', va='center', fontsize=10, fontweight='bold')

# Decode tokens
for i in range(n_tokens):
    start = queue_time + ttft + i * tpot
    color = '#2ecc71' if i % 2 == 0 else '#27ae60'
    ax.barh(y, tpot, left=start, height=height, color=color, alpha=0.7, edgecolor='black', linewidth=0.5)

# Labels
decode_start = queue_time + ttft
decode_end = decode_start + n_tokens * tpot
ax.annotate('', xy=(decode_start, y-0.25), xytext=(decode_end, y-0.25),
            arrowprops=dict(arrowstyle='<->', color='red', linewidth=2))
ax.text((decode_start + decode_end)/2, y-0.35, f'Decode Phase ({n_tokens} tokens)', 
        ha='center', fontsize=11, color='red')

# E2E latency
ax.annotate('', xy=(0, y+0.25), xytext=(decode_end, y+0.25),
            arrowprops=dict(arrowstyle='<->', color='purple', linewidth=2))
ax.text(decode_end/2, y+0.35, f'E2E Latency = {decode_end:.2f}s', 
        ha='center', fontsize=12, color='purple', fontweight='bold')

# TTFT marker
ax.axvline(x=queue_time + ttft, color='blue', linestyle='--', alpha=0.5)
ax.text(queue_time + ttft + 0.02, y + 0.5, f'TTFT = {queue_time + ttft:.2f}s', 
        fontsize=10, color='blue')

ax.set_xlabel('Time (seconds)', fontsize=12)
ax.set_title('Anatomy of an LLM Request', fontsize=15)
ax.set_yticks([])
ax.set_xlim(-0.05, decode_end + 0.1)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"Queue time:     {queue_time*1000:.0f} ms")
print(f"TTFT:           {(queue_time + ttft)*1000:.0f} ms")
print(f"TPOT:           {tpot*1000:.0f} ms")
print(f"Output tokens:  {n_tokens}")
print(f"E2E Latency:    {decode_end*1000:.0f} ms")

---

## 2. The Latency-Throughput Trade-off

### The Fundamental Trade-off

Increasing batch size:
- **Increases throughput**: More tokens processed per GPU-second
- **Increases per-request latency**: Each request gets a smaller share of compute

This is because during decode, all requests in the batch share the same weight-reading bandwidth:

$$\text{TPOT}(\text{batch\_size}) \approx \frac{\text{model\_size}}{\text{bandwidth}} \quad (\text{for small batches, memory-bound})$$

But throughput increases linearly:
$$\text{Throughput}(\text{batch\_size}) = \frac{\text{batch\_size}}{\text{TPOT}(\text{batch\_size})}$$

In [ ]:
def simulate_latency_throughput(
    model_size_gb: float = 14.0,
    bandwidth_gbs: float = 2000.0,  # GB/s
    max_batch_size: int = 256,
    peak_tflops: float = 312.0,
    d_model: int = 4096,
    n_layers: int = 32,
    dtype_bytes: int = 2
):
    """
    Simulate how TPOT and throughput change with batch size.
    
    Accounts for both memory-bound and compute-bound regimes.
    """
    batch_sizes = np.arange(1, max_batch_size + 1)
    
    tpot_ms = []
    throughputs = []
    
    for bs in batch_sizes:
        # Memory time: reading model weights
        mem_time = model_size_gb / bandwidth_gbs  # seconds
        
        # Compute time: 2 * params * batch_size FLOPs (approximate)
        total_flops = 2 * (model_size_gb / dtype_bytes * 1e9) * bs
        compute_time = total_flops / (peak_tflops * 1e12)  # seconds
        
        # Total time is max of memory and compute (whichever is bottleneck)
        total_time = max(mem_time, compute_time)
        
        # TPOT is total time divided by batch size
        tpot = total_time / bs * 1000  # ms per token
        tpot_ms.append(tpot)
        
        # Throughput
        throughput = bs / total_time
        throughputs.append(throughput)
    
    return batch_sizes, np.array(tpot_ms), np.array(throughputs)


# Simulate for A100
batch_sizes, tpot, throughput = simulate_latency_throughput(
    model_size_gb=14.0, bandwidth_gbs=2000.0, peak_tflops=312.0
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# TPOT vs batch size
axes[0].plot(batch_sizes, tpot, 'b-', linewidth=2)
axes[0].set_xlabel('Batch Size', fontsize=12)
axes[0].set_ylabel('TPOT (ms)', fontsize=12)
axes[0].set_title('Per-Token Latency vs Batch Size', fontsize=14)
axes[0].axhline(y=50, color='red', linestyle='--', alpha=0.5, label='50ms target')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Throughput vs batch size
axes[1].plot(batch_sizes, throughput, 'g-', linewidth=2)
axes[1].set_xlabel('Batch Size', fontsize=12)
axes[1].set_ylabel('Throughput (tokens/s)', fontsize=12)
axes[1].set_title('Throughput vs Batch Size', fontsize=14)
axes[1].grid(True, alpha=0.3)

# Latency vs throughput (the trade-off curve)
axes[2].plot(throughput, tpot, 'r-', linewidth=2)
axes[2].set_xlabel('Throughput (tokens/s)', fontsize=12)
axes[2].set_ylabel('TPOT (ms)', fontsize=12)
axes[2].set_title('Latency-Throughput Trade-off', fontsize=14)
axes[2].axhline(y=50, color='orange', linestyle='--', alpha=0.5, label='50ms TPOT target')

# Mark some batch sizes
for bs in [1, 8, 32, 128]:
    idx = bs - 1
    axes[2].plot(throughput[idx], tpot[idx], 'ko', markersize=8)
    axes[2].annotate(f'B={bs}', xy=(throughput[idx], tpot[idx]),
                     xytext=(throughput[idx]+100, tpot[idx]+1),
                     fontsize=9)

axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 3. Simulating a Real Serving Workload

In practice, requests arrive randomly and have variable prompt/output lengths. Let's simulate this.

In [ ]:
@dataclass
class Request:
    """A simulated LLM request."""
    id: int
    arrival_time: float      # when the request arrives
    prompt_length: int       # number of input tokens
    output_length: int       # number of output tokens to generate
    
    # Filled during simulation
    start_time: float = 0.0       # when processing begins
    first_token_time: float = 0.0 # when first token is generated
    end_time: float = 0.0         # when last token is generated
    token_times: list = field(default_factory=list)  # time of each token
    
    @property
    def ttft(self) -> float:
        return self.first_token_time - self.arrival_time
    
    @property
    def tpot(self) -> float:
        if len(self.token_times) <= 1:
            return 0
        return (self.end_time - self.first_token_time) / (len(self.token_times) - 1)
    
    @property
    def e2e_latency(self) -> float:
        return self.end_time - self.arrival_time
    
    @property
    def queue_time(self) -> float:
        return self.start_time - self.arrival_time


def generate_workload(
    n_requests: int = 100,
    arrival_rate: float = 10.0,  # requests per second
    prompt_mean: int = 500,
    prompt_std: int = 200,
    output_mean: int = 200,
    output_std: int = 100,
) -> List[Request]:
    """
    Generate a synthetic workload with Poisson arrivals.
    """
    requests = []
    current_time = 0.0
    
    for i in range(n_requests):
        inter_arrival = np.random.exponential(1.0 / arrival_rate)
        current_time += inter_arrival
        
        prompt_len = max(10, int(np.random.normal(prompt_mean, prompt_std)))
        output_len = max(1, int(np.random.normal(output_mean, output_std)))
        
        requests.append(Request(
            id=i,
            arrival_time=current_time,
            prompt_length=prompt_len,
            output_length=output_len
        ))
    
    return requests


# Generate workload
workload = generate_workload(n_requests=200, arrival_rate=5.0)

# Show workload statistics
prompt_lens = [r.prompt_length for r in workload]
output_lens = [r.output_length for r in workload]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist([r.arrival_time for r in workload[:50]], bins=30, color='#3498db', alpha=0.7)
axes[0].set_xlabel('Time (s)')
axes[0].set_title('Request Arrival Times (first 50)')

axes[1].hist(prompt_lens, bins=30, color='#2ecc71', alpha=0.7)
axes[1].set_xlabel('Prompt Length (tokens)')
axes[1].set_title(f'Prompt Length Distribution (mean={np.mean(prompt_lens):.0f})')

axes[2].hist(output_lens, bins=30, color='#e74c3c', alpha=0.7)
axes[2].set_xlabel('Output Length (tokens)')
axes[2].set_title(f'Output Length Distribution (mean={np.mean(output_lens):.0f})')

plt.tight_layout()
plt.show()

In [ ]:
def simulate_serving(
    requests: List[Request],
    max_batch_size: int = 32,
    prefill_speed: float = 10000.0,  # tokens/s for prefill
    decode_base_ms: float = 7.0,     # base decode time per step (ms)
    decode_per_request_ms: float = 0.5,  # additional decode time per request in batch
) -> List[Request]:
    """
    Simulate continuous batching serving.
    
    Simple model:
    - Prefill takes prompt_length / prefill_speed seconds
    - Decode step takes (decode_base + batch_size * decode_per_request) ms
    - New requests can join the batch between decode steps
    """
    completed = []
    pending = list(requests)  # requests waiting to be scheduled
    active = []  # requests currently being decoded
    current_time = 0.0
    
    while pending or active:
        # Add new requests that have arrived
        newly_arrived = []
        remaining_pending = []
        for req in pending:
            if req.arrival_time <= current_time and len(active) + len(newly_arrived) < max_batch_size:
                newly_arrived.append(req)
            else:
                remaining_pending.append(req)
        pending = remaining_pending
        
        # Prefill new requests
        for req in newly_arrived:
            prefill_time = req.prompt_length / prefill_speed
            req.start_time = current_time
            req.first_token_time = current_time + prefill_time
            req.token_times = [req.first_token_time]
            current_time += prefill_time  # simplified: sequential prefill
            active.append(req)
        
        if not active:
            # Jump to next request arrival
            if pending:
                current_time = pending[0].arrival_time
            continue
        
        # Decode step for all active requests
        batch_size = len(active)
        step_time = (decode_base_ms + batch_size * decode_per_request_ms) / 1000.0
        current_time += step_time
        
        # Update each request
        finished = []
        for req in active:
            req.token_times.append(current_time)
            if len(req.token_times) >= req.output_length:
                req.end_time = current_time
                finished.append(req)
        
        for req in finished:
            active.remove(req)
            completed.append(req)
    
    return completed


# Run simulation
completed = simulate_serving(workload, max_batch_size=32)

print(f"Completed {len(completed)} requests")
print(f"Time span: {completed[0].arrival_time:.1f}s to {completed[-1].end_time:.1f}s")

In [ ]:
# Analyze and plot metrics

ttfts = [r.ttft * 1000 for r in completed]  # ms
tpots = [r.tpot * 1000 for r in completed]  # ms
e2e_latencies = [r.e2e_latency * 1000 for r in completed]  # ms
queue_times = [r.queue_time * 1000 for r in completed]  # ms

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# TTFT distribution
axes[0, 0].hist(ttfts, bins=40, color='#3498db', alpha=0.7, edgecolor='white')
axes[0, 0].axvline(x=np.percentile(ttfts, 50), color='green', linestyle='-', linewidth=2, label=f'P50={np.percentile(ttfts, 50):.0f}ms')
axes[0, 0].axvline(x=np.percentile(ttfts, 99), color='red', linestyle='--', linewidth=2, label=f'P99={np.percentile(ttfts, 99):.0f}ms')
axes[0, 0].set_xlabel('TTFT (ms)')
axes[0, 0].set_title('Time to First Token Distribution')
axes[0, 0].legend()

# TPOT distribution
axes[0, 1].hist(tpots, bins=40, color='#2ecc71', alpha=0.7, edgecolor='white')
axes[0, 1].axvline(x=np.percentile(tpots, 50), color='green', linestyle='-', linewidth=2, label=f'P50={np.percentile(tpots, 50):.0f}ms')
axes[0, 1].axvline(x=np.percentile(tpots, 99), color='red', linestyle='--', linewidth=2, label=f'P99={np.percentile(tpots, 99):.0f}ms')
axes[0, 1].set_xlabel('TPOT (ms)')
axes[0, 1].set_title('Time Per Output Token Distribution')
axes[0, 1].legend()

# E2E latency distribution
axes[1, 0].hist(e2e_latencies, bins=40, color='#e74c3c', alpha=0.7, edgecolor='white')
axes[1, 0].axvline(x=np.percentile(e2e_latencies, 50), color='green', linestyle='-', linewidth=2, label=f'P50={np.percentile(e2e_latencies, 50):.0f}ms')
axes[1, 0].axvline(x=np.percentile(e2e_latencies, 99), color='red', linestyle='--', linewidth=2, label=f'P99={np.percentile(e2e_latencies, 99):.0f}ms')
axes[1, 0].set_xlabel('E2E Latency (ms)')
axes[1, 0].set_title('End-to-End Latency Distribution')
axes[1, 0].legend()

# Queue time over time
axes[1, 1].scatter([r.arrival_time for r in completed], queue_times, 
                    alpha=0.5, s=10, c='#9b59b6')
axes[1, 1].set_xlabel('Arrival Time (s)')
axes[1, 1].set_ylabel('Queue Wait Time (ms)')
axes[1, 1].set_title('Queue Time Over Time')

plt.tight_layout()
plt.show()

# Print summary statistics
print("\nMetrics Summary")
print("=" * 50)
for name, data in [('TTFT', ttfts), ('TPOT', tpots), ('E2E Latency', e2e_latencies)]:
    print(f"\n{name}:")
    print(f"  Mean:  {np.mean(data):>8.1f} ms")
    print(f"  P50:   {np.percentile(data, 50):>8.1f} ms")
    print(f"  P90:   {np.percentile(data, 90):>8.1f} ms")
    print(f"  P95:   {np.percentile(data, 95):>8.1f} ms")
    print(f"  P99:   {np.percentile(data, 99):>8.1f} ms")
    print(f"  Max:   {np.max(data):>8.1f} ms")

---

## 4. Impact of Batch Size on Metrics

Let's run the simulation with different maximum batch sizes and observe the trade-offs.

In [ ]:
# Compare different max batch sizes

batch_sizes_to_test = [1, 4, 8, 16, 32, 64, 128]
results = {}

for bs in batch_sizes_to_test:
    # Regenerate workload (same seed)
    np.random.seed(42)
    workload = generate_workload(n_requests=200, arrival_rate=5.0)
    completed = simulate_serving(workload, max_batch_size=bs)
    
    ttfts = [r.ttft * 1000 for r in completed]
    tpots = [r.tpot * 1000 for r in completed]
    e2e_lats = [r.e2e_latency * 1000 for r in completed]
    
    total_time = max(r.end_time for r in completed) - min(r.arrival_time for r in completed)
    total_tokens = sum(r.output_length for r in completed)
    throughput = total_tokens / total_time
    
    results[bs] = {
        'ttft_p50': np.percentile(ttfts, 50),
        'ttft_p99': np.percentile(ttfts, 99),
        'tpot_p50': np.percentile(tpots, 50),
        'tpot_p99': np.percentile(tpots, 99),
        'e2e_p50': np.percentile(e2e_lats, 50),
        'e2e_p99': np.percentile(e2e_lats, 99),
        'throughput': throughput
    }

# Plot results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

bs_list = list(results.keys())

# TTFT
axes[0, 0].plot(bs_list, [results[b]['ttft_p50'] for b in bs_list], 'b-o', label='P50', linewidth=2)
axes[0, 0].plot(bs_list, [results[b]['ttft_p99'] for b in bs_list], 'r-o', label='P99', linewidth=2)
axes[0, 0].set_xlabel('Max Batch Size')
axes[0, 0].set_ylabel('TTFT (ms)')
axes[0, 0].set_title('TTFT vs Batch Size')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# TPOT
axes[0, 1].plot(bs_list, [results[b]['tpot_p50'] for b in bs_list], 'b-o', label='P50', linewidth=2)
axes[0, 1].plot(bs_list, [results[b]['tpot_p99'] for b in bs_list], 'r-o', label='P99', linewidth=2)
axes[0, 1].set_xlabel('Max Batch Size')
axes[0, 1].set_ylabel('TPOT (ms)')
axes[0, 1].set_title('TPOT vs Batch Size')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Throughput
axes[1, 0].bar(range(len(bs_list)), [results[b]['throughput'] for b in bs_list],
               color='#2ecc71', alpha=0.8)
axes[1, 0].set_xticks(range(len(bs_list)))
axes[1, 0].set_xticklabels(bs_list)
axes[1, 0].set_xlabel('Max Batch Size')
axes[1, 0].set_ylabel('Throughput (tokens/s)')
axes[1, 0].set_title('Throughput vs Batch Size')
axes[1, 0].grid(True, alpha=0.3)

# Latency-Throughput trade-off
throughputs = [results[b]['throughput'] for b in bs_list]
p50_lats = [results[b]['tpot_p50'] for b in bs_list]
axes[1, 1].plot(throughputs, p50_lats, 'r-o', linewidth=2, markersize=10)
for bs, tp, lat in zip(bs_list, throughputs, p50_lats):
    axes[1, 1].annotate(f'B={bs}', xy=(tp, lat), xytext=(tp+50, lat+0.5), fontsize=9)
axes[1, 1].set_xlabel('Throughput (tokens/s)')
axes[1, 1].set_ylabel('TPOT P50 (ms)')
axes[1, 1].set_title('Latency-Throughput Trade-off')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 5. Service Level Objectives (SLOs)

### 5.1 What are SLOs?

SLOs define the acceptable performance bounds for a serving system:

| Metric | Chatbot SLO | Batch API SLO | Code Completion SLO |
|--------|------------|---------------|---------------------|
| TTFT P99 | < 500 ms | < 5 s | < 200 ms |
| TPOT P99 | < 100 ms | < 200 ms | < 50 ms |
| E2E P99 | < 10 s | < 60 s | < 3 s |
| Throughput | > 500 tok/s | > 5000 tok/s | > 100 tok/s |

### 5.2 Why P99 Matters

Mean latency hides outliers. If you have 100 users and your P99 TTFT is 5s, that means on average, one user out of every 100 waits more than 5 seconds just for the first token!

In [ ]:
# SLO compliance analysis

def check_slo_compliance(
    requests: List[Request],
    slo_ttft_ms: float = 500,
    slo_tpot_ms: float = 100,
    slo_e2e_ms: float = 10000
):
    """Check what percentage of requests meet SLO targets."""
    ttfts = [r.ttft * 1000 for r in requests]
    tpots = [r.tpot * 1000 for r in requests]
    e2e_lats = [r.e2e_latency * 1000 for r in requests]
    
    ttft_pass = sum(1 for t in ttfts if t <= slo_ttft_ms) / len(ttfts) * 100
    tpot_pass = sum(1 for t in tpots if t <= slo_tpot_ms) / len(tpots) * 100
    e2e_pass = sum(1 for t in e2e_lats if t <= slo_e2e_ms) / len(e2e_lats) * 100
    
    all_pass = sum(1 for t, p, e in zip(ttfts, tpots, e2e_lats) 
                   if t <= slo_ttft_ms and p <= slo_tpot_ms and e <= slo_e2e_ms) / len(requests) * 100
    
    return {
        'ttft_compliance': ttft_pass,
        'tpot_compliance': tpot_pass,
        'e2e_compliance': e2e_pass,
        'overall_compliance': all_pass
    }


# Check SLO compliance for different batch sizes
print("SLO Compliance Analysis")
print(f"SLO: TTFT <= 500ms, TPOT <= 100ms, E2E <= 10s")
print("=" * 70)
print(f"{'Batch Size':<15} {'TTFT %':<10} {'TPOT %':<10} {'E2E %':<10} {'All %':<10}")
print("-" * 70)

for bs in batch_sizes_to_test:
    np.random.seed(42)
    workload = generate_workload(n_requests=200, arrival_rate=5.0)
    completed = simulate_serving(workload, max_batch_size=bs)
    compliance = check_slo_compliance(completed)
    
    print(f"{bs:<15} {compliance['ttft_compliance']:<10.1f} {compliance['tpot_compliance']:<10.1f} "
          f"{compliance['e2e_compliance']:<10.1f} {compliance['overall_compliance']:<10.1f}")

---

## 6. Benchmarking Methodology

### How to Measure LLM Serving Performance Correctly

Common mistakes in benchmarking:

1. **Using synchronous requests**: This doesn't test the system under load
2. **Fixed prompt/output length**: Real workloads have variable lengths
3. **Measuring only mean**: Tail latency (P99) matters more
4. **Not warming up**: First few requests may have cold-start effects
5. **Not controlling for arrival pattern**: Bursty vs steady traffic changes behavior

In [ ]:
class MetricsDashboard:
    """
    A metrics dashboard for LLM serving analysis.
    """
    
    def __init__(self, requests: List[Request]):
        self.requests = requests
        self.ttfts = [r.ttft * 1000 for r in requests]  # ms
        self.tpots = [r.tpot * 1000 for r in requests]  # ms
        self.e2e_lats = [r.e2e_latency * 1000 for r in requests]  # ms
        self.queue_times = [r.queue_time * 1000 for r in requests]  # ms
    
    def summary(self):
        """Print a summary of all metrics."""
        total_time = max(r.end_time for r in self.requests) - min(r.arrival_time for r in self.requests)
        total_input_tokens = sum(r.prompt_length for r in self.requests)
        total_output_tokens = sum(r.output_length for r in self.requests)
        
        print("LLM Serving Metrics Dashboard")
        print("=" * 60)
        print(f"Total requests:        {len(self.requests)}")
        print(f"Total time:            {total_time:.1f} s")
        print(f"Request rate:          {len(self.requests) / total_time:.1f} req/s")
        print(f"Input throughput:      {total_input_tokens / total_time:.0f} tok/s")
        print(f"Output throughput:     {total_output_tokens / total_time:.0f} tok/s")
        print(f"Total throughput:      {(total_input_tokens + total_output_tokens) / total_time:.0f} tok/s")
        print()
        
        for name, data in [('TTFT', self.ttfts), ('TPOT', self.tpots), 
                           ('E2E Latency', self.e2e_lats), ('Queue Time', self.queue_times)]:
            print(f"{name}:")
            print(f"  Mean={np.mean(data):.0f}ms  P50={np.percentile(data, 50):.0f}ms  "
                  f"P90={np.percentile(data, 90):.0f}ms  P99={np.percentile(data, 99):.0f}ms  "
                  f"Max={max(data):.0f}ms")
    
    def plot(self):
        """Generate a comprehensive dashboard."""
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle('LLM Serving Metrics Dashboard', fontsize=16, fontweight='bold')
        
        # 1. TTFT CDF
        sorted_ttft = np.sort(self.ttfts)
        cdf = np.arange(1, len(sorted_ttft)+1) / len(sorted_ttft)
        axes[0,0].plot(sorted_ttft, cdf, 'b-', linewidth=2)
        axes[0,0].axhline(y=0.99, color='red', linestyle='--', alpha=0.5, label='P99')
        axes[0,0].set_xlabel('TTFT (ms)')
        axes[0,0].set_ylabel('CDF')
        axes[0,0].set_title('TTFT Cumulative Distribution')
        axes[0,0].legend()
        axes[0,0].grid(True, alpha=0.3)
        
        # 2. TPOT CDF
        sorted_tpot = np.sort(self.tpots)
        cdf = np.arange(1, len(sorted_tpot)+1) / len(sorted_tpot)
        axes[0,1].plot(sorted_tpot, cdf, 'g-', linewidth=2)
        axes[0,1].axhline(y=0.99, color='red', linestyle='--', alpha=0.5, label='P99')
        axes[0,1].set_xlabel('TPOT (ms)')
        axes[0,1].set_ylabel('CDF')
        axes[0,1].set_title('TPOT Cumulative Distribution')
        axes[0,1].legend()
        axes[0,1].grid(True, alpha=0.3)
        
        # 3. E2E Latency vs Output Length
        axes[0,2].scatter([r.output_length for r in self.requests], self.e2e_lats,
                          alpha=0.5, s=15, c='#e74c3c')
        axes[0,2].set_xlabel('Output Length (tokens)')
        axes[0,2].set_ylabel('E2E Latency (ms)')
        axes[0,2].set_title('E2E Latency vs Output Length')
        axes[0,2].grid(True, alpha=0.3)
        
        # 4. Throughput over time (sliding window)
        window_size = 5.0  # seconds
        arrival_times = sorted([r.arrival_time for r in self.requests])
        time_points = np.arange(arrival_times[0], arrival_times[-1], 1.0)
        throughputs = []
        for t in time_points:
            tokens_in_window = sum(r.output_length for r in self.requests 
                                   if t - window_size <= r.end_time <= t)
            throughputs.append(tokens_in_window / window_size)
        axes[1,0].plot(time_points, throughputs, 'b-', linewidth=1.5)
        axes[1,0].set_xlabel('Time (s)')
        axes[1,0].set_ylabel('Throughput (tok/s)')
        axes[1,0].set_title(f'Throughput Over Time ({window_size}s window)')
        axes[1,0].grid(True, alpha=0.3)
        
        # 5. TTFT vs Prompt Length
        axes[1,1].scatter([r.prompt_length for r in self.requests], self.ttfts,
                          alpha=0.5, s=15, c='#3498db')
        axes[1,1].set_xlabel('Prompt Length (tokens)')
        axes[1,1].set_ylabel('TTFT (ms)')
        axes[1,1].set_title('TTFT vs Prompt Length')
        axes[1,1].grid(True, alpha=0.3)
        
        # 6. Percentile comparison bars
        percentiles = [50, 90, 95, 99]
        x = np.arange(len(percentiles))
        width = 0.25
        
        ttft_percs = [np.percentile(self.ttfts, p) for p in percentiles]
        tpot_percs = [np.percentile(self.tpots, p) for p in percentiles]
        
        axes[1,2].bar(x - width/2, ttft_percs, width, label='TTFT', color='#3498db', alpha=0.8)
        axes[1,2].bar(x + width/2, tpot_percs, width, label='TPOT', color='#2ecc71', alpha=0.8)
        axes[1,2].set_xticks(x)
        axes[1,2].set_xticklabels([f'P{p}' for p in percentiles])
        axes[1,2].set_ylabel('Latency (ms)')
        axes[1,2].set_title('Latency Percentiles')
        axes[1,2].legend()
        axes[1,2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()


# Create dashboard
np.random.seed(42)
workload = generate_workload(n_requests=500, arrival_rate=10.0)
completed = simulate_serving(workload, max_batch_size=32)

dashboard = MetricsDashboard(completed)
dashboard.summary()
dashboard.plot()

---

## Exercises

### Exercise 1: Build a Metrics Dashboard Simulator

Extend the simulator to support:
1. Variable arrival rates (simulate load spikes)
2. Priority queuing (premium vs free users)
3. Request timeout handling

In [ ]:
def simulate_with_load_spikes(
    duration_s: float = 60.0,
    base_rate: float = 5.0,
    spike_rate: float = 50.0,
    spike_start: float = 20.0,
    spike_duration: float = 10.0,
    max_batch_size: int = 32,
    timeout_ms: float = 30000  # 30s timeout
):
    """
    Exercise: Simulate serving with a traffic spike.
    
    Generate requests at base_rate normally, but during
    [spike_start, spike_start + spike_duration], increase to spike_rate.
    
    Analyze how the spike affects:
    - Queue times
    - TTFT
    - Timeout rate
    - Recovery time after spike
    """
    # YOUR CODE HERE
    pass

# Test:
# simulate_with_load_spikes()

### Exercise 2: Optimal Batch Size Finder

Write a function that finds the optimal batch size given SLO constraints.

In [ ]:
def find_optimal_batch_size(
    arrival_rate: float = 10.0,
    slo_ttft_p99_ms: float = 500.0,
    slo_tpot_p99_ms: float = 100.0,
    batch_sizes_to_test: list = None
) -> int:
    """
    Exercise: Find the maximum batch size that meets SLO constraints
    while maximizing throughput.
    
    Run simulations with different batch sizes and find the one that:
    1. Meets TTFT P99 SLO
    2. Meets TPOT P99 SLO
    3. Maximizes throughput
    """
    # YOUR CODE HERE
    pass

# Test:
# optimal_bs = find_optimal_batch_size()
# print(f"Optimal batch size: {optimal_bs}")

### Exercise 3: SLO Compliance Checker

Implement a function that monitors a stream of completed requests and reports
SLO compliance in real time, including per-window compliance rates.

In [ ]:
def slo_compliance_checker(
    completed_requests: list,
    window_size_s: float = 10.0,
    slo_ttft_p99_ms: float = 500.0,
    slo_tpot_p99_ms: float = 100.0,
    slo_e2e_p99_ms: float = 10000.0,
) -> dict:
    """
    Exercise: Implement an SLO compliance checker.

    Given a list of completed Request objects, compute per-window compliance:

    Steps:
    1. Divide the time range into windows of `window_size_s` seconds
       based on each request's arrival_time.
    2. For each window, compute P99 of TTFT, TPOT, and E2E latency.
    3. Determine whether each window meets the SLO targets.
    4. Return a summary dict with:
       - "windows": list of dicts, each with:
           - "start_time": float
           - "end_time": float
           - "num_requests": int
           - "ttft_p99_ms": float
           - "tpot_p99_ms": float
           - "e2e_p99_ms": float
           - "ttft_ok": bool  (True if ttft_p99 <= slo_ttft_p99_ms)
           - "tpot_ok": bool
           - "e2e_ok": bool
           - "all_ok": bool   (True if all three SLOs met)
       - "overall_compliance_pct": float  (% of windows where all_ok is True)
       - "total_requests": int
       - "total_windows": int

    Hints:
    - Use request.arrival_time to assign requests to windows.
    - Use np.percentile() with the 99th percentile.
    - Skip windows with fewer than 2 requests (not enough data for P99).
    """
    # TODO: Determine the time range from completed_requests
    # min_time = ...
    # max_time = ...

    # TODO: Create windows of window_size_s seconds
    # windows = []

    # TODO: For each window, filter requests by arrival_time
    # TODO: Compute P99 for TTFT, TPOT, E2E (in ms)
    # TODO: Check each metric against its SLO target

    # TODO: Compute overall compliance percentage

    # TODO: Return the summary dict
    pass

# Test with the simulated workload:
# np.random.seed(42)
# workload = generate_workload(n_requests=200, arrival_rate=5.0)
# completed = simulate_serving(workload, max_batch_size=32)
# report = slo_compliance_checker(completed)
# print(f"Overall SLO compliance: {report['overall_compliance_pct']:.1f}%")
# for w in report['windows'][:5]:
#     status = "PASS" if w['all_ok'] else "FAIL"
#     print(f"  [{status}] Window {w['start_time']:.0f}-{w['end_time']:.0f}s: "
#           f"TTFT_P99={w['ttft_p99_ms']:.0f}ms, TPOT_P99={w['tpot_p99_ms']:.0f}ms")

---

## Summary

### Key Takeaways

1. **TTFT** (Time to First Token) is dominated by prefill computation and queue wait time. Critical for user experience.

2. **TPOT** (Time Per Output Token) is determined by batch size and model size. Increases with batch size but each additional request adds less overhead.

3. **Throughput vs Latency**: Fundamental trade-off. Larger batches increase throughput but also increase per-request latency.

4. **Tail latency (P99, P999)** matters more than mean for production systems. One bad experience out of 100 is significant.

5. **SLOs** should be set based on use case: interactive chatbots need tight latency bounds; batch processing can tolerate higher latency.

6. **Benchmarking** requires realistic workloads with variable lengths, proper warm-up, and percentile reporting.

### What's Next

In **Chapter 1.6: Batching Strategies**, we'll dive deep into the different batching strategies (static, dynamic, continuous) and understand why continuous batching is the breakthrough that enables high-throughput LLM serving.